# BigBasket Category Performance Diagnostic — Part 3
## Python / Pandas Cleaning & Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv("orders_raw.csv")
products = pd.read_csv("products.csv")
print("Raw shape:", orders.shape)
print(orders.head())

In [ ]:
# Remove exact duplicates and normalize inconsistent categorical values
orders = orders.drop_duplicates().copy()
orders["city"] = orders["city"].astype(str).str.strip().str.title()
orders["category"] = orders["category"].astype(str).str.strip().str.title()
orders["amount_inr"] = pd.to_numeric(orders["amount_inr"], errors="coerce")

print("Rows after cleaning:", len(orders))
print("Distinct cities:", orders["city"].nunique())
print("Distinct categories:", orders["category"].nunique())
print("Missing amount_inr:", orders["amount_inr"].isna().sum())
print("Cancelled/Pending rating nulls left unfilled:", orders.loc[orders["status"].isin(["Cancelled","Pending"]), "rating"].isna().sum())

### IQR outlier detection and capping
Q1 and Q3 are computed only on Delivered orders with non-null revenue. Values above the upper fence are capped, not dropped.

In [ ]:
delivered_mask = (orders["status"] == "Delivered") & orders["amount_inr"].notna()
Q1 = orders.loc[delivered_mask, "amount_inr"].quantile(0.25)
Q3 = orders.loc[delivered_mask, "amount_inr"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
capped_rows = delivered_mask & (orders["amount_inr"] > upper_fence)
print(f"Q1 = ₹{Q1:,.2f}")
print(f"Q3 = ₹{Q3:,.2f}")
print(f"IQR = ₹{IQR:,.2f}")
print(f"Upper fence = ₹{upper_fence:,.2f}")
print("Rows capped =", int(capped_rows.sum()))
orders.loc[delivered_mask, "amount_inr"] = orders.loc[delivered_mask, "amount_inr"].clip(upper=upper_fence)

In [ ]:
# Parse dates and create derived columns
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["month"] = orders["order_date"].dt.month
orders["month_name"] = orders["order_date"].dt.month_name()
orders["revenue_per_unit"] = orders["amount_inr"] / orders["quantity"]
orders["is_delivered"] = orders["status"] == "Delivered"

# Rating nulls for Cancelled/Pending remain unfilled intentionally.

In [ ]:
# Category and supplier revenue
clean_delivered = orders[orders["is_delivered"] & orders["amount_inr"].notna()].copy()
category_revenue = clean_delivered.groupby("category")["amount_inr"].sum().sort_values(ascending=False)
print("Top category:", category_revenue.index[0], f"₹{category_revenue.iloc[0]:,.2f}")

orders_with_supplier = orders.merge(products[["product_id", "supplier"]], on="product_id", how="left")
supplier_revenue = orders_with_supplier[orders_with_supplier["is_delivered"] & orders_with_supplier["amount_inr"].notna()].groupby("supplier")["amount_inr"].sum().sort_values(ascending=False)
print("Top supplier:", supplier_revenue.index[0], f"₹{supplier_revenue.iloc[0]:,.2f}")
print("Part 1 top category match:", category_revenue.index[0] == "Household Essentials")
print("Part 1 top supplier match:", supplier_revenue.index[0] == "HomeEssentials Traders")

In [ ]:
# Chart 1: category revenue
plt.figure()
category_revenue.plot(kind="bar")
plt.title(f"Household Essentials leads category revenue at ₹{category_revenue.iloc[0]:,.0f}")
plt.xlabel("Category"); plt.ylabel("Revenue (INR)"); plt.xticks(rotation=35, ha="right"); plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: monthly Delivered revenue trend
monthly_revenue = clean_delivered.groupby(["month", "month_name"])["amount_inr"].sum().reset_index().sort_values("month")
plt.figure()
plt.plot(monthly_revenue["month_name"], monthly_revenue["amount_inr"], marker="o")
peak = monthly_revenue.loc[monthly_revenue["amount_inr"].idxmax()]
plt.title(f"{peak['month_name']} has the highest monthly Delivered revenue")
plt.xlabel("Month"); plt.ylabel("Revenue (INR)"); plt.xticks(rotation=25); plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: supplier revenue
supplier_revenue.head(8).sort_values().plot(kind="barh")
plt.title(f"{supplier_revenue.index[0]} generates the most supplier revenue")
plt.xlabel("Revenue (INR)"); plt.ylabel("Supplier"); plt.tight_layout(); plt.show()

### 1
**What:** Household Essentials generated **₹20,910.00**, or **24.7%** of cleaned Delivered revenue.  
**Why it matters:** It is the largest revenue contributor in the cleaned analysis.  
**Next step:** Review its product-level and supplier-level contribution to identify the main drivers of this result.

### 2
**What:** **HomeEssentials Traders** generated the highest cleaned supplier revenue at **₹20,910.00**.  
**Why it matters:** Supplier-level contribution can help explain where category revenue is concentrated.  
**Next step:** Compare this supplier's revenue with the remaining suppliers and inspect its associated categories/products.

### 3
**What:** **May** recorded the highest cleaned Delivered monthly revenue at **₹16,668.50**.  
**Why it matters:** The monthly peak identifies a period where demand or order mix was strongest in this dataset.  
**Next step:** Break that month down by category and supplier to identify the contributors to the peak.